In [2]:
!pip install ifcopenshell

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 MB 19.8 MB/s eta 0:00:00


In [3]:
# Script searches the ifc for alle IfcColourRgb values and changes them to the provided RGB colour value.
# Example: change all colours of a demolition model to yellow
import ifcopenshell
import os

# Define the file paths
ifc_file_path = '/content/LU31_AR_MD_ME_ABBRU.ifc'

# Define the new color in RGB
new_color_rgb = (255, 255, 128)

# Automatically generate the output file path
file_name, file_extension = os.path.splitext(ifc_file_path)
output_ifc_file_path = f"{file_name}_modified{file_extension}"


# Function to convert RGB to IFC color
def rgb_to_ifc_color(rgb_color):
  if not isinstance(rgb_color, tuple) or len(rgb_color) != 3:
    raise ValueError("Input must be a tuple of three integers.")
  if not all(0 <= c <= 255 for c in rgb_color):
      raise ValueError("RGB values must be between 0 and 255.")

  return (rgb_color[0] / 255.0, rgb_color[1] / 255.0, rgb_color[2] / 255.0)


# Search for lines containing "IFCCOLOUR" in the IFC file
found_lines = []
try:
    with open(ifc_file_path, 'r') as f:
        print(f"Searching for lines containing 'IFCCOLOUR' in {ifc_file_path}...")
        found_lines = [line.strip() for line in f if "IFCCOLOUR" in line]

    if not found_lines:
        print("\nNo lines containing 'IFCCOLOUR' found in the file.")

except FileNotFoundError:
    print(f"Error: The file {ifc_file_path} was not found.")
    # Exit or handle the error appropriately if the file is not found
    exit()
except Exception as e:
    print(f"An error occurred: {e}")
    # Exit or handle the error appropriately if an error occurs
    exit()


# Extract IfcColourRgb Entity IDs
ifc_color_ids = []
for line in found_lines:
    if line.startswith('#') and '=IFCCOLOURRGB' in line:
        try:
            entity_id_str = line.split('=')[0][1:]
            entity_id = int(entity_id_str)
            ifc_color_ids.append(entity_id)
        except (ValueError, IndexError) as e:
            print(f"Could not extract entity ID from line: {line} - {e}")

print(f"Extracted {len(ifc_color_ids)} IfcColourRgb entity IDs: {ifc_color_ids}")

# Load the IFC file
try:
    ifc_file = ifcopenshell.open(ifc_file_path)
except Exception as e:
    print(f"Error loading IFC file: {e}")
    # Exit or handle the error appropriately if the file cannot be loaded
    exit()


# Change the color of IfcColourRgb entities
new_color = rgb_to_ifc_color(new_color_rgb)

changed_count = 0
for entity_id in ifc_color_ids:
    try:
        color_entity = ifc_file.by_id(entity_id)
        if color_entity and color_entity.is_a("IfcColourRgb"):
            color_entity.Red = new_color[0]
            color_entity.Green = new_color[1]
            color_entity.Blue = new_color[2]
            changed_count += 1
        else:
            print(f"Warning: Entity with ID {entity_id} not found or is not IfcColourRgb.")
    except Exception as e:
        print(f"Error processing entity ID {entity_id}: {e}")


print(f"Changed color for {changed_count} IfcColourRgb entities to {new_color}.")

# Save the modified IFC file
try:
    ifc_file.write(output_ifc_file_path)
    print(f"Modified IFC file saved to: {output_ifc_file_path}")
except Exception as e:
    print(f"Error saving modified IFC file: {e}")

Searching for lines containing 'IFCCOLOUR' in /content/LU31_AR_MD_ME_ABBRU.ifc...
Extracted 5 IfcColourRgb entity IDs: [90, 98, 2333, 2737, 3099]
Changed color for 5 IfcColourRgb entities to (1.0, 1.0, 0.5019607843137255).
Modified IFC file saved to: /content/LU31_AR_MD_ME_ABBRU_modified.ifc
